# 🚢 Task 2 — Titanic Dataset Analysis
**Maincrafts Technology | Data Analysis with Python Internship**

**Intern:** Mukund Rajpurohit | **Intern ID:** MT5153

---

## 📌 Objective
Analyze the Titanic dataset to find survival patterns based on:
- Gender (Male vs Female)
- Passenger Class (1st, 2nd, 3rd)
- Age Groups

**Tools Used:** Python, Pandas, Matplotlib, Seaborn

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Libraries imported successfully!')

## 📂 Step 2 — Load the Titanic Dataset
Loading directly from the web — no need to download manually!

In [ ]:
# Load Titanic dataset directly from URL (no Kaggle login needed)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(f'✅ Dataset loaded successfully!')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 🔍 Step 3 — Explore the Dataset

In [ ]:
# Basic info about the dataset
print('=== DATASET INFO ===')
df.info()

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df.describe()

In [ ]:
# Check for missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_df[missing_df['Missing Count'] > 0])

## 🧹 Step 4 — Data Cleaning
Handling missing values in Age, Cabin, and Embarked columns.

In [ ]:
# 1. Fill missing Age with median age (better than mean — less affected by outliers)
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f'✅ Missing Age values filled with median age: {median_age}')

# 2. Drop Cabin column — too many missing values (77%)
df.drop(columns=['Cabin'], inplace=True)
print('✅ Cabin column dropped (77% missing values)')

# 3. Fill missing Embarked with mode (most frequent value)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
print(f'✅ Missing Embarked filled with mode: {df["Embarked"].mode()[0]}')

# Verify no missing values remain
print(f'\n📊 Missing values after cleaning: {df.isnull().sum().sum()}')
print(f'✅ Dataset is clean and ready for analysis!')

In [ ]:
# Create Age Groups for analysis
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (60+)']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

print('✅ Age groups created:')
print(df['AgeGroup'].value_counts())

## 📊 Step 5 — Analysis & Key Questions

### ❓ Q1: Who survived more — Males or Females?

In [ ]:
# Survival count by gender
survival_gender = df.groupby('Sex')['Survived'].agg(['sum', 'count', 'mean']).reset_index()
survival_gender.columns = ['Gender', 'Survived', 'Total', 'Survival Rate']
survival_gender['Survival Rate'] = (survival_gender['Survival Rate'] * 100).round(2)
survival_gender['Did Not Survive'] = survival_gender['Total'] - survival_gender['Survived']

print('=== SURVIVAL BY GENDER ===')
print(survival_gender.to_string(index=False))

print(f'\n✅ INSIGHT: Females had a {survival_gender[survival_gender["Gender"]=="female"]["Survival Rate"].values[0]}% survival rate')
print(f'   Males had only a {survival_gender[survival_gender["Gender"]=="male"]["Survival Rate"].values[0]}% survival rate')
print('   → "Women and children first" policy is clearly reflected in the data!')

### ❓ Q2: Did Passenger Class affect survival chances?

In [ ]:
# Survival by passenger class
survival_class = df.groupby('Pclass')['Survived'].agg(['sum', 'count', 'mean']).reset_index()
survival_class.columns = ['Class', 'Survived', 'Total', 'Survival Rate']
survival_class['Survival Rate'] = (survival_class['Survival Rate'] * 100).round(2)
survival_class['Class'] = survival_class['Class'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})

print('=== SURVIVAL BY PASSENGER CLASS ===')
print(survival_class.to_string(index=False))

print(f'\n✅ INSIGHT: 1st Class passengers had the highest survival rate!')
print('   Wealthier passengers had better access to lifeboats (closer to deck).')
print('   3rd Class passengers had the lowest survival rate — reflecting class inequality.')

### ❓ Q3: What was the survival rate by Age Group?

In [ ]:
# Survival by age group
survival_age = df.groupby('AgeGroup')['Survived'].agg(['sum', 'count', 'mean']).reset_index()
survival_age.columns = ['Age Group', 'Survived', 'Total', 'Survival Rate']
survival_age['Survival Rate'] = (survival_age['Survival Rate'] * 100).round(2)

print('=== SURVIVAL BY AGE GROUP ===')
print(survival_age.to_string(index=False))

print('\n✅ INSIGHT: Children (0-12) had the highest survival rate.')
print('   Senior passengers (60+) had one of the lowest survival rates.')

## 📈 Step 6 — Data Visualizations

In [ ]:
# ── CHART 1: Survival by Gender ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — survival count by gender
sns.barplot(x='Sex', y='Survived', data=df, palette=['#FF6B9D', '#4ECDC4'],
            estimator=lambda x: sum(x)/len(x)*100, ax=axes[0])
axes[0].set_title('Survival Rate by Gender (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Gender', fontsize=12)
axes[0].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0].set_xticklabels(['Female', 'Male'], fontsize=11)

# Add value labels on bars
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Count chart
survived_gender = df.groupby(['Sex', 'Survived']).size().unstack()
survived_gender.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#51CF66'],
                     edgecolor='white', width=0.6)
axes[1].set_title('Survival Count by Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(['Female', 'Male'], rotation=0, fontsize=11)
axes[1].legend(['Did Not Survive', 'Survived'], fontsize=10)

plt.suptitle('📊 Gender vs Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart1_survival_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 1: Survival by Gender saved!')

In [ ]:
# ── CHART 2: Survival by Passenger Class ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival rate by class
sns.barplot(x='Pclass', y='Survived', data=df,
            palette=['#FFD700', '#C0C0C0', '#CD7F32'],
            estimator=lambda x: sum(x)/len(x)*100, ax=axes[0])
axes[0].set_title('Survival Rate by Passenger Class (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Passenger Class', fontsize=12)
axes[0].set_ylabel('Survival Rate (%)', fontsize=12)
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], fontsize=11)

for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Survival count by class
survived_class = df.groupby(['Pclass', 'Survived']).size().unstack()
survived_class.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#51CF66'],
                    edgecolor='white', width=0.6)
axes[1].set_title('Survival Count by Passenger Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0, fontsize=11)
axes[1].legend(['Did Not Survive', 'Survived'], fontsize=10)

plt.suptitle('📊 Passenger Class vs Survival Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_survival_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 2: Survival by Class saved!')

In [ ]:
# ── CHART 3: Histogram of Passenger Ages ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution histogram
axes[0].hist(df['Age'], bins=20, color='#4ECDC4', edgecolor='white', linewidth=0.8)
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean Age: {df["Age"].mean():.1f}')
axes[0].axvline(df['Age'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median Age: {df["Age"].median():.1f}')
axes[0].set_title('Distribution of Passenger Ages', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=12)
axes[0].set_ylabel('Number of Passengers', fontsize=12)
axes[0].legend(fontsize=10)

# Survived vs Not Survived age distribution
df[df['Survived']==1]['Age'].hist(ax=axes[1], bins=20, alpha=0.7,
                                   color='#51CF66', label='Survived', edgecolor='white')
df[df['Survived']==0]['Age'].hist(ax=axes[1], bins=20, alpha=0.7,
                                   color='#FF6B6B', label='Did Not Survive', edgecolor='white')
axes[1].set_title('Age Distribution: Survived vs Not Survived', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('📊 Passenger Age Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart3_age_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 3: Age Histogram saved!')

In [ ]:
# ── CHART 4 (BONUS): Survival by Age Group ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

age_surv = df.groupby('AgeGroup')['Survived'].mean() * 100
bars = ax.bar(age_surv.index, age_surv.values,
              color=['#74C0FC', '#51CF66', '#FFD43B', '#FF922B', '#FF6B6B'],
              edgecolor='white', width=0.5)

for bar, val in zip(bars, age_surv.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

ax.set_title('📊 Survival Rate by Age Group (%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Age Group', fontsize=12)
ax.set_ylabel('Survival Rate (%)', fontsize=12)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('chart4_survival_by_agegroup.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 4: Survival by Age Group saved!')

## 📝 Step 7 — Summary of Insights

In [ ]:
total     = len(df)
survived  = df['Survived'].sum()
overall_rate = round(survived / total * 100, 2)

female_rate = round(df[df['Sex']=='female']['Survived'].mean() * 100, 2)
male_rate   = round(df[df['Sex']=='male']['Survived'].mean() * 100, 2)
c1_rate = round(df[df['Pclass']==1]['Survived'].mean() * 100, 2)
c2_rate = round(df[df['Pclass']==2]['Survived'].mean() * 100, 2)
c3_rate = round(df[df['Pclass']==3]['Survived'].mean() * 100, 2)

print('=' * 55)
print('        TITANIC DATASET — FINAL SUMMARY REPORT')
print('=' * 55)
print(f'  Total Passengers   : {total}')
print(f'  Total Survived     : {survived} ({overall_rate}%)')
print(f'  Total Not Survived : {total - survived} ({100 - overall_rate}%)')
print('-' * 55)
print('  SURVIVAL BY GENDER:')
print(f'    Female Survival Rate : {female_rate}%  ✅')
print(f'    Male Survival Rate   : {male_rate}%  ❌')
print('-' * 55)
print('  SURVIVAL BY PASSENGER CLASS:')
print(f'    1st Class : {c1_rate}%  🥇')
print(f'    2nd Class : {c2_rate}%  🥈')
print(f'    3rd Class : {c3_rate}%  🥉')
print('-' * 55)
print('  KEY INSIGHTS:')
print('  1. Females survived at 3x the rate of males')
print('     (Women & children first policy)')
print('  2. 1st class passengers survived more than')
print('     3rd class — wealth/location on ship mattered')
print('  3. Children had highest survival among age groups')
print('  4. Most passengers were 20-35 years old')
print('=' * 55)
print('  ✅ Task 2 Complete — Mukund Rajpurohit (MT5153)')
print('=' * 55)